> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내**
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 06. 프롬프트 테스트·평가·개선</b></font>


## <font color='Darkorange'><b>[ Chapter 02 ]</b></font> Chapter 02. 평가와 개선

해당 챕터는 **주피터 노트북 실습 기반**으로 진행됩니다.  
실습 시작 전 아래 설정을 반드시 실행해주세요.


### ⚙️ <font color='#007A45'><b>[ 실습 전 ]</b></font> Part6 Chapter 02 실습 전 프로젝트 셋업
>  ✅ 아래 **실습 전 가상환경을 활성화하고, 프로젝트 셋업**을 완료한 후 본 실습을 진행해주세요.

> ⚠️ 실습 진행 중 에러가 발생하거나, 세션이 종료되어 런타임이 재시작된 경우, 이 블럭을 항상 다시 실행해주세요.


```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

In [ ]:
# 1. uv 설치 
%pip install uv

In [ ]:
# 2. 필요한 패키지 설치 (최초 1회만)
%uv pip install -r ../requirements.txt

#### 실습 진행을 위한 API KEY 세팅

실습을 진행하기 위해서는 각 AI서비스들의 API Key를 발급 및 세팅 해야합니다.

LangSmith, Gemini, Claude, Chat GPT API Key를 모두 발급하셨다면, 아래 코드 블럭을 실행하여 API Key를 세팅해봅시다.


In [ ]:
# LangSmith & OpenAI Key 설정 (.env 파일 → 환경변수)
import os, re
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경변수 확인
print("✅ 환경변수 설정 완료 (등록된 키만 활성화됨)")
print("-" * 40)
for k in [
    "OPENAI_API_KEY",
    "LANGSMITH_API_KEY",
    "GOOGLE_API_KEY",
    "ANTHROPIC_API_KEY",
    "TAVILY_API_KEY",
]:
    v = os.getenv(k)
    print(f"{k}: {v[:6] + '*'*10 if v else v}")

#### 실습 진행을 위해 모델 호출 함수 정의 세팅

In [ ]:
import datetime
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import AIMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate

from langchain.chat_models import init_chat_model

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call

from typing import List, Dict, Annotated, Any
from datetime import datetime
from pydantic import BaseModel, Field


MODELS = {
    "openai": init_chat_model("openai:gpt-4o-mini"),
    "claude": init_chat_model("anthropic:claude-3-5-haiku-20241022"),
    "gemini": init_chat_model("google_genai:gemini-2.5-flash"),
}


def _run_base_chain(model_key: str, input_data, system_prompt=None, **kwargs):
    """지정된 모델로 실행. 문자열 또는 메시지 리스트 입력 지원."""
    try:
        model = MODELS.get(model_key)
        if not model:
            return f"Error: {model_key} not found"

        if kwargs:
            model = model.with_config(**kwargs)

        if isinstance(input_data, list):
            messages = input_data
        else:
            messages = []
            if system_prompt:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": str(input_data)})

        response = model.invoke(messages)
        return response.content

    except Exception as e:
        return f"Error: {str(e)}"


def start_chat_session(model_key: str, system_prompt: str = ""):
    """멀티턴 대화 시작."""
    model_obj = MODELS.get(model_key)
    if not model_obj:
        print(f"Error: {model_key} not found")
        return

    model_name = getattr(model_obj, "model_name", "Unknown")
    print(f"[{model_key.upper()} - {model_name}] 채팅 시작 (exit로 종료)")
    print("-" * 50)

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    while True:
        user_input = input(">>> You: ")
        if user_input.lower() in ["exit", "q", "quit"]:
            print("👋 종료합니다.")
            break

        messages.append({"role": "user", "content": user_input})
        response = _run_base_chain(model_key, messages)

        print(f"🤖 AI: {response}\n")
        messages.append({"role": "assistant", "content": response})


def run_openai_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain("openai", system_prompt, user_input, **kwargs)


def run_claude_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain("claude", system_prompt, user_input, **kwargs)


def run_gemini_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain("gemini", system_prompt, user_input, **kwargs)

실행 예시를 알아봅시다.

In [ ]:
# 실행 예시
system_prompt = "You are a concise and helpful AI assistant."
user_input = "너에 대해 소개해줘!"

# OpenAI 모델 실행
print("# OpenAI Result:")
print(run_openai_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Claude 모델 실행
print("# Claude Result:")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Gemini 모델 실행
print("# Gemini Result:")
print(run_gemini_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

파라미터 수정 예시를 알아봅시다.

In [ ]:
# 1. Temperature (창의성 조절) -- 필요 시 주석 해제하여 실행
print("# OpenAI (temperature=0.8)")
# print(run_openai_chain(system_prompt=system_prompt, user_input=user_input, temperature=0.8))

print("(모델 파라미터 변경 코드는 주석 처리되어 있습니다)")
print("-" * 40)


# 2. Top-p -- 필요 시 주석 해제하여 실행 -- 필요 시 주석 해제하여 실행
print("# Claude (top_p=0.7)")
# print(run_claude_chain(system_prompt=system_prompt, user_input=user_input, top_p=0.7))

print("(모델 파라미터 변경 코드는 주석 처리되어 있습니다)")
print("-" * 40)


# 3. Max tokens (출력 길이 제한) — 필요 시 주석 해제하여 실행
print("# Gemini (max_output_tokens=100)")
# print(run_gemini_chain(system_prompt=system_prompt, user_input=user_input, max_output_tokens=100))

print("(모델 파라미터 변경 코드는 주석 처리되어 있습니다)")
print("-" * 40)


# 4. 모델 이름 교체 (필요 시 주석 해제하여 실행)
print("[4. Model Swap]")
# 주의: 아래 코드를 실행하면 이후 'openai' 키는 gpt-3.5-turbo로 고정됩니다.
# 교체를 원하시면 아래 주석을 해제하고 실행하세요.

# MODELS["openai"] = init_chat_model("gpt-3.5-turbo", model_provider="openai")
# print(run_openai_chain(system_prompt, user_input))

print("(모델 변경 코드는 주석 처리되어 있습니다)")
print("-" * 40)

# 5. 복수 파라미터 동시 변경 — 필요 시 주석 해제하여 실행
print("# Claude (temperature=0.9, top_p=0.95)")
# print(run_claude_chain(system_prompt=system_prompt, user_input=user_input, temperature=0.9, top_p=0.95))

print("(모델 파라미터 변경 코드는 주석 처리되어 있습니다)")
print("-" * 40)

#### 실습 확인을 위한 LangSmith 추적 세팅 함수

사용자가 실습 기록을 구분하기 위해 LangSmith 프로젝트명을 입력하면 되는 함수입니다.

입력한 이름으로 LangSmith 대시보드에 실행 내역이 저장됩니다.
(예: prompt-course, rag-lab1, myproject-001 등)


```python
# 프로젝트명을 변수로 바로 지정
LANGSMITH_PROJECT = "prompt-course"

# 함수 호출로 환경변수 등록
setup_langsmith(LANGSMITH_PROJECT)
```



In [ ]:
# LangSmith 설정 함수 (프로젝트명만 입력받아 환경변수 등록)


def setup_langsmith(project_name: str):
    """
    LangSmith 관련 환경변수를 등록하는 함수입니다.
    이미 등록된 LANGSMITH_API_KEY를 사용하며,
    project_name 변수로 LangSmith 프로젝트명을 지정할 수 있습니다.
    """
    LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
    LANGSMITH_TRACING = "true"

    os.environ.update(
        {
            "LANGSMITH_PROJECT": project_name,
            "LANGSMITH_ENDPOINT": LANGSMITH_ENDPOINT,
            "LANGSMITH_TRACING": LANGSMITH_TRACING,
        }
    )

    print("✅ LangSmith 설정 완료")
    print(f"- PROJECT : {project_name}")
    print(f"- ENDPOINT: {LANGSMITH_ENDPOINT}")
    print(f"- TRACING : {LANGSMITH_TRACING}")

#### 최종 실습 준비

In [ ]:
setup_langsmith("prompt-course")

### <font color='green'><b>[ 실습 ] </b></font> 02. 프롬프트 Response Emulation




**실습문제 2: 프롬프트 Function Calling for Scoring**
1. 각 기준의 ‘name’과 ‘description’에 해당하는 프롬프트를 수정하세요.
2. 수정함에 따라 Score가 어떻게 나오는지 확인하세요.

---

*평가 기준*
1. appropriateness
2. clarity
3. empathy
4. completeness


#### **1️⃣ Load Dataset**


실습에서 사용할 데이터를 불러와서 데이터를 확인해보도록 하겠습니다.  
먼저, 공유드린 `dialog.csv` 파일을 주피터 노트북에 업로드 해주겠습니다.  

---

```
💁‍♀️
말씀드린 것처럼, 주피터 노트북 세션으로 관리되고 있어서
세션이 만료될때마다 파일을 다시 업로드해야하니 참고해주세요.

이제 업로드한 파일을 불러와 어떻게 구성되어 있는지 확인해봅시다!
```

In [ ]:
import pandas as pd

df = pd.read_csv("data/dialog.csv")
df.head()

**데이터 확인**

이번 실습에서는 편의상 데이터셋이 single-turn으로만 구성되어 있습니다.

업로드한 발화 데이터가 single-turn으로 구성되어 있는지 확인해봅시다!

---

> Single turn은 대화가 한번에 끝나는 - 한번의 질문과 한번의 답변으로 이루어진 간단한 대화가 해당됩니다.

<center><img src="https://drive.google.com/uc?id=1WKh70LpjojzDvPNcUuRABBdoOBYgb-7M" width=300></center>

<br>

In [ ]:
for key, value in dict(df.iloc[0]).items():
    print(f"{key}: \n{value}\n")

In [ ]:
print(f"데이터셋은 {len(df)}개의 single-turn 발화 데이터로 구성되어 있습니다.")

데이터셋이 single-turn으로 구성된 것을 확인하였습니다.

이제 이 발화 데이터를 평가해봅시다!

#### 2️⃣ 평가 환경 설정: Function

---

call 함수 정의

프롬프트 평가 기준 8개 중, 아래 4개의 기준에 대해 평가하는 function call 을 작성해보겠습니다.

1. appropriateness
2. clarity
3. empathy
4. completeness


function calling을 위한 구성은 아래와 같습니다.  
크게 `tools`, `messages`, `model`이 필요합니다.
<br>

``` python
tools = [
    {
        "type": "function",
        "function": {
            "name": "function_name",
            "description": "function_description",
            "parameters": {
                "type": "object",
                "properties": {
                    "property_name": {
                        "type": "string",
                        "description": "property_description",
                    },
                },
                "required": ["property_name_required"],
                "additionalProperties": False,
            },
        }
    }
]

messages = [
    {"role": "system", "content": "system message"},
    {"role": "user", "content": "user message"}
]

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)
```

<hr>

<br>
<br>

**2.1.1 tools 셋업**

전달받은 입력을 어떠한 기준으로 어떻게 분류할지 정의하는 곳이 tools 입니다.
 
이해를 위해 좀 더 쉽게 설명해보면 아래와 같습니다.

<hr>

```
function
- name : 함수명 (예. 감정 분류 함수)
- description: 함수 설명 (예. 유저의 입력을 보고 긍정, 부정을 분류합니다.)
- parameters:
    - type: object
    - properties: 각 파라미터에 대한 세부 속성
        - name: 파라미터 이름 (예. 감정)
            - type: 해당 파라미터의 데이터 타입 (예. string, boolean)
            - description: 이 파라미터의 분류 기준 등을 정의(예. input의 감정이 긍정적이라면 positive, 부정적이라면 negative로 분류합니다.)
    - required: 필수로 받아야 할 파라미터들 목록
    - additionalProperties: 정의된 것 이외의 추가 파라미터를 받을 수 있는지 여부
```

In [ ]:
from typing import List, Dict


class function_generator:
    """
    function calling을 위한 function 정의 함수.
    field명, field의 설명, 각 field 하위의 property를 받아 tools를 정의합니다.

    field_name: str,
    description: str,
    properties_context: List[Dict]
        {property_name} : {
            "type" : "boolean",
            "description": str
        }
    """

    def __init__(
        self, field_name: str, description: str, properties_context: List[Dict]
    ):
        self.function = {
            "name": f"eval_{field_name}",
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False,
            },
        }

        for field_property in properties_context:
            property_name, property_description = (
                field_property["name"],
                field_property["description"],
            )
            self.function["parameters"]["properties"][property_name] = {
                "type": "boolean",
                "description": property_description,
            }

        self.function["parameters"]["required"] = list(
            self.function["parameters"]["properties"].keys()
        )

    def get_function(self):
        return self.function

<br>

평가 기준 1: **appropriateness**

---

👉 만족스러운 응답은 대화의 맥락을 준수하며 사용자의 쿼리에 관련성이 있고 적합한 정보를 제공합니다.

아래 코드는 `appropriateness`를 검증하기 위한 function call 셋업입니다.



In [ ]:
# -- appropriateness 를 검증하기 위한 function call 셋업

import json

field_name = "appropriateness"
description = "Satisfying responses adhere to the context of the conversation and provide information that is relevant and suitable for the user's query."
properties_context = [
    {
        "name": "specification",
        "description": "Does the response align with the context and specificity of the user's query?",
    },
    {
        "name": "clear_language",
        "description": "Is the language and terminology used suitable for the user's evident level of understanding or expertise?",
    },
    {
        "name": "language_tone",
        "description": "Does the response maintain a consistent and neutral tone throughout its content?",
    },
    {
        "name": "proper_statements",
        "description": "Are any statements or phrases given by the companion that could be considered offensive, inaccurate, or inappropriate for the user's query?",
    },
]

# 정의한 field_name, description, properties_context를 전달해서 function을 정의함
appropriateness_function = function_generator(
    field_name, description, properties_context
)
print(json.dumps(appropriateness_function.get_function(), indent=2))

<br>

평가 기준 2: **clarity**

---

👉 만족스러운 답변은 명확하고 이해하기 쉽습니다. 지나치게 복잡한 언어, 전문 용어, 복잡한 문장 구조를 피하여 사용자가 쉽게 정보를 이해할 수 있도록 합니다.

아래 코드는 `clarity`를 검증하기 위한 function call 셋업입니다.



In [ ]:
# -- clarity 를 검증하기 위한 function call 셋업

field_name = "clarity"
description = "Satisfying responses are clear and easy to understand. The companion avoids overly complex language, jargon, or convoluted sentence structures, ensuring that users can grasp the information effortlessly."
properties_context = [
    {
        "name": "structure",
        "description": "Does the companion present its response in a structured and organized manner, allowing for easy comprehension?",
    },
    {
        "name": "clear_language",
        "description": "If the companion uses any specialized terminology, does it provide clear definitions or explanations to ensure user comprehension? ",
    },
    {
        "name": "clear_introduction",
        "description": "Does the companion avoid lengthy or convoluted sentences that could confuse the user?",
    },
    {
        "name": "ambiguity",
        "description": "Are any potential ambiguities in the response clarified without prompting from the user?",
    },
]


clarity_function = function_generator(field_name, description, properties_context)
# print(json.dumps(clarity_function.get_function(), indent=2))

<br>

평가 기준 3: **empathy**

---

👉 특정 상황에서는 공감을 표현하면 사용자 만족도가 높아질 수 있습니다. 동반자는 감정을 갖고 있지 않을 수도 있지만 필요할 경우 동정적인 언어를 사용할 수 있습니다.

아래 코드는 `empathy`를 검증하기 위한 function call 셋업입니다.



In [ ]:
# -- empathy 를 검증하기 위한 function call 셋업

field_name = "empathy"
description = "In certain situations, showing empathy can enhance user satisfaction. While companion may not possess emotions, it can use sympathetic language when appropriate."
properties_context = [
    {
        "name": "emotion",
        "description": "Does the companion recognize when a user is expressing emotions and respond with appropriate empathetic language?",
    },
    {
        "name": "support",
        "description": "In situations where a user is upset or frustrated, does the companion maintain a calm and supportive tone?",
    },
    {
        "name": "understanding",
        "description": "Does the companion’s response convey understanding and validation of the user’s feelings, even if it cannot experience emotions itself?",
    },
]


empathy_function = function_generator(field_name, description, properties_context)
# print(json.dumps(empathy_function.get_function(), indent=2))

<br>

평가 기준 4: **completeness**

---

👉 사용자는 쿼리의 모든 측면을 포괄하는 포괄적인 답변을 높이 평가합니다. 동반자는 모호하거나 부분적인 답변을 해서는 안 되며, 대신 관련된 모든 사항을 언급해야 합니다.


아래 코드는 `completeness`를 검증하기 위한 function call 셋업입니다.



In [ ]:
# -- completeness 를 검증하기 위한 function call 셋업

field_name = "completeness"
description = "Users appreciate comprehensive answers that cover all aspects of their queries. The companion should not provide vague or partial responses but instead, address all relevant points."
properties_context = [
    {
        "name": "thoroughness",
        "description": "Does the companion's response thoroughly address all parts of the user's query?",
    },
    {
        "name": "acknowledgement",
        "description": "If the user’s question has multiple components, does the companion acknowledge and answer each one individually?",
    },
    {
        "name": "depth",
        "description": "Does the companion provide a sufficient depth of information to ensure the user’s query is fully addressed, rather than providing a superficial answer? ",
    },
]


completeness_function = function_generator(field_name, description, properties_context)
# print(json.dumps(completeness_function.get_function(), indent=2))

In [ ]:
# -- 정의한 function을 tools list에 취합

tools = []

fields = [
    appropriateness_function.get_function(),
    clarity_function.get_function(),
    empathy_function.get_function(),
    completeness_function.get_function(),
]

for field in fields:
    tools.append({"type": "function", "function": field})


print(json.dumps(tools, indent=2))

**2.1.2 messages 셋업**

In [ ]:
from enum import Enum


# -- 평가 기준
FIELDS = ["appropriateness", "clarity", "empathy", "completeness"]


# -- datafram을 읽어서 정해진 포맷으로 반환하는 함수
def dialog_generator(single_turn: dict) -> str:
    return f"""User: {single_turn['user']}
Assistant: {single_turn['assistant']}
"""


# -- system prompt
def system_prompt_generator(field, dialog):
    return f"""You are tasked with evaluating the {field} displayed by companion in the dialogue between user and the companion.
{field} is a multi-faceted trait, encompassing several attributes.
Your objective is to read the conversation attentively and answer the following questions with either a "yes" or a "no".

[dialog]
{dialog}"""

**2.2 function calling request**




In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

from pandas import DataFrame
import json
from typing import Dict, Any


def prompt_eval_request(target_dialog: DataFrame):
    dialog = dialog_generator(dict(target_dialog))
    field_list = ", ".join(FIELDS)

    system_prompt = system_prompt_generator(field_list, dialog)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
        ]
    )

    model = init_chat_model("openai:gpt-4o-mini").bind_tools(tools)
    # tool_model = model.bind_tools(tools)

    chain = prompt | model
    ai_msg = chain.invoke({})

    result = {}

    for tool_call in ai_msg.tool_calls or []:
        field_name = tool_call["name"].split("_")[1]
        result[field_name] = tool_call["args"]

    return result

✔️ **함수들이 설정이 잘 되었는지 한번의 데이터로 테스트 해보겠습니다.**

In [ ]:
results = []
result = prompt_eval_request(df.iloc[0])
results.append(result)

**✔️ 요청 결과를 확인해봅시다!**

In [ ]:
import json

print(json.dumps(result, indent=2))

이렇게 나온 결과물들을 기존 dataframe과 함께 볼 수 있도록 **시각화 함수를 정의**해봅시다.

In [ ]:
import pandas as pd
from pandas import DataFrame


def combine_result_to_df(dialog_df: DataFrame, results: List):

    eval_col_names = [
        "appropriateness/specification",
        "appropriateness/clear_language",
        "appropriateness/language_tone",
        "appropriateness/proper_statements",
        "clarity/structure",
        "clarity/clear_language",
        "clarity/clear_introduction",
        "clarity/ambiguity",
        "empathy/emotion",
        "empathy/support",
        "empathy/understanding",
        "completeness/thoroughness",
        "completeness/acknowledgement",
        "completeness/depth",
    ]

    columns = ["user", "assistant"] + eval_col_names
    result_df = pd.DataFrame(columns=columns)
    result_df["user"] = dialog_df["user"]
    result_df["assistant"] = dialog_df["assistant"]

    for idx in range(len(results)):
        result = results[idx]

        for eval_col_name in eval_col_names:
            eval_field, subclass = eval_col_name.split("/")
            result_df.loc[idx, eval_col_name] = result[eval_field][subclass]

    return result_df

위에서 정의한 시각화 함수를 통해 전달받은 한개의 결과가 추가된 dataframe을 확인해봅시다!

In [ ]:
result_df = combine_result_to_df(df, results)
result_df[0:12]

이제 모든 준비가 완료되었습니다!  
전체 데이터셋 평가를 시작해봅시다!! 👏👏👏

---

<br>

> 🚨
아래 코드 블럭을 실행하면
총 13건의 데이터셋에 대해 평가를 요청하는 api에 request 하게 됩니다.
>
> 의도하지 않은 api 사용을 막아두기 위해
8번째 줄에 "break"로 한번의 요청만 보내도록 설정해두었습니다.
>
> 우선 한건만 테스트하여 어떤 결과가 반환되는지 확인해봅시다.
>
> (전체적으로 확인이 완료되었다면
break 줄을 주석처리 하여 전체 13건의 데이터를 테스트해봅시다.)

In [ ]:
from tqdm import tqdm

results = []

for idx in tqdm(range(len(df)), desc="prompt eval ..."):
    print(dict(df.iloc[idx]))  # <-- request한 발화를 확인하고 싶다면 주석 해제

    result = prompt_eval_request(df.iloc[idx])
    results.append(result)
    # break  # <-- 이 부분 주석처리 하기

In [ ]:
result_df = combine_result_to_df(df, results)
result_df.head(12)

**2.3 결과 취합 후 확인하기**

In [ ]:
# True를 1로, False를 0으로 변환
score_df = result_df.replace({True: 1, False: 0})

# 필요한 컬럼들만 선택
columns_to_sum = score_df.columns[2:]

# 행 별 점수 합계 계산
score_df["score_sum"] = score_df[columns_to_sum].sum(axis=1)


# 결과 출력
score_df[["user", "assistant", "score_sum"]]

### <font color='green'><b>[ 실습 ] </b></font> 03. 프롬프트 평가 실습하기: Langgraph



> 📍 랭그래프 기반 프롬프트 평가 코드를 실행 및 프롬프트를 변경하며
각 노드의 결과가 어떻게 달라지는지 테스트해보세요.

**실습문제3: Prompt Optimization with LangGraph: 5-Step Process**

- Process:

    Step 1. Analyze the Prompt Given by User  
    Step 2. Human Feedback  
    Step 3. Test the Prompt   
    Step 4. Optimize the Prompt    
    Step 5. Evaluate the Prompt

#### Set UP

In [ ]:
from typing import Annotated, TypedDict, Dict
import operator
import re
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END


# Initialization
def get_llm(model_name: str, api_keys: Dict[str, str] = None, **kwargs) -> ChatOpenAI:
    return ChatOpenAI(model=model_name, **kwargs)

# parser = RegexParser(
#     regex=r"revised prompt:\s*(.+)",
#     output_keys=["revised_prompt"]
# )


parser = StrOutputParser()

# Set up the GraphState
class GraphState(TypedDict):
    """
    A TypedDict class representing the state of the prompt optimization graph.

    Attributes:
        prompt (Annotated[list, operator.add]): List containing the prompts being optimized
        ai_feedback (Annotated[list, operator.add]): List containing AI-generated feedback on prompts
        human_feedback (Annotated[list, operator.add]): List containing human feedback on prompts
    """
    prompt: Annotated[list, operator.add]
    ai_feedback: Annotated[list, operator.add]
    human_feedback: Annotated[list, operator.add]

#### The Code of Prompt Optimization with LangGraph

In [ ]:
# Initialize the LLMs
gpt = get_llm(model_name="gpt-4o-mini", temperature=0.0)
test_gpt = get_llm(model_name="gpt-4o-mini", temperature=0.0)


# 1st node: analyze_prompt
def analyze_prompt(state: GraphState, llm: ChatOpenAI = gpt) -> GraphState:
    """
    Analyzes the given prompt and revise it for improvement.

    Args:
        state (GraphState): Current state contains the prompt to analyze
        llm (ChatOpenAI): Language model( with temperature: 0.0 ) used to generate feedback and improvements

    Returns:
        GraphState: Updated state with AI feedback and revised prompt
    """
    latest_prompt = state["prompt"][0]

    print("[ Here is the user's prompt ]")
    print(latest_prompt)

    prompt = PromptTemplate.from_template(
        """
    Your task is to analyze user's prompt.

    1. Extract the user's intent. And write down a single word focused on 'action ascription'.
    2. Check your output is aligned with the user's intent.
    3. Suggest me some improvements for the prompt.
    4. Revise my prompt to ensure it includes two focal elements: examples and ample context within the prompt.
    5. Your output should be within 4 sentences.
    Format:
        (1) user's intent:
        (2) some improvement:
        (3) revised prompt: *don't answer to the user's prompt.



    -------->
    User's prompt:
    {prompt}
    -------->
    """
    )

    chain = prompt | llm | StrOutputParser()

    response = chain.invoke({"prompt": latest_prompt})

    match = re.search(
        r'revised_prompt["\s]*:\s*["\']([^"\']+)["\']', response, re.IGNORECASE
    )
    revised_prompt = match.group(1) if match else response

    return {"prompt": [revised_prompt], "ai_feedback": [response]}


# 2nd node: human
def human_feedback(state: GraphState) -> GraphState:
    """
    Collects feedback from the user.

    Args:
        state (GraphState): Current state of the graph

    Returns:
        GraphState: Update state with human feedback
    """
    print("[ Here is the proposed prompt ] \n", state["prompt"][-1])
    user_input = input("(Press 'q' or 'quit' to quit)")

    return {"human_feedback": [user_input]}


# 3rd node: test_prompt
def test_optimized_prompt(state: GraphState, llm: ChatOpenAI = test_gpt) -> GraphState:
    """
    Tests both previous and current prompts, comparing their responses.

    Args:
        state (GraphState): Current state containing both prompts
        llm (ChatOpenAI): Language model( with temperature: 0.7 ) used to generate feedback

    Returns:
        GraphState: Updated state with comparison of responses
    """
    previous_prompt = state["prompt"][-2]  # Get previous prompt
    current_prompt = state["prompt"][-1]  # Get latest prompt

    # Get responses for both prompts
    previous_response = llm.invoke(previous_prompt)
    revised_response = llm.invoke(current_prompt)

    # Compare the responses
    comparison_prompt = f"""
    Compare these two responses:

    PREVIOUS PROMPT: {previous_prompt}
    PREVIOUS RESPONSE: {previous_response.content}

    REVISED PROMPT: {current_prompt}
    REVISED RESPONSE: {revised_response.content}

    Please analyze:
    1. Key differences in responses.
    2. Summarize the differences within 1 sentence.
    3. tell me which one is better: Previous or Revised?
    """

    analysis = llm.invoke(comparison_prompt)

    return {
        "ai_feedback": [
            f"Previous Prompt Response:\n{previous_response.content}\n\n"
            f"Revised Prompt Response:\n{revised_response.content}\n\n"
            f"Comparative Analysis:\n{analysis.content}"
        ]
    }


# 3th node: optimize_prompt
def optimize_prompt(state: GraphState, llm: ChatOpenAI = gpt) -> GraphState:
    """
    Optimizes the prompt based on your feedback and the user's feedback.

    Args:
        state (GraphState): Current state contains the prompt, AI feedback(your feedback), and the user's feedback
        llm (ChatOpenAI): Language model( with temperature: 0.0 ) used to optimize prompt based on feedback

    Returns:
        GraphState: Updated state with optimized prompt
    """
    latest_prompt = state["prompt"][-1]
    latest_ai_feedback = state["ai_feedback"][-1]
    latest_human_feedback = state["human_feedback"][-1]

    if latest_human_feedback == "q" or latest_human_feedback == "quit":
        latest_human_feedback = ""
        return {"prompt": [latest_prompt]}

    prompt = PromptTemplate.from_template(
        """
    Your task is to optimize the user's prompt.
    Follow these instructions:

    Here is the prompt before improvement:
    {prompt}

    Here is previous feedback generated by AI:
    {ai_feedback}

    Here is Human's follow-up feedback:
    {human_feedback}

    Write down the improved prompt. Only present the revised prompt without any additional comments.
    """
    )

    chain = prompt | llm | parser

    response = chain.invoke(
        {
            "prompt": latest_prompt,
            "ai_feedback": latest_ai_feedback,
            "human_feedback": latest_human_feedback,
        }
    )

    match = re.search(
        r'revised_prompt["\s]*:\s*["\']([^"\']+)["\']', response, re.IGNORECASE
    )
    revised_prompt = match.group(1) if match else response

    return {"prompt": [revised_prompt]}

    # return {"prompt": [parsed["revised_prompt"]]}


# 4th node: evaluate_prompt
def evaluate_prompt(state: GraphState, llm: ChatOpenAI = gpt) -> GraphState:
    """
    Evaluates the optimized prompt by comparing it to the previous prompt.

    Args:
        state (GraphState): Current state contains the previous prompt and the optimized prompt
        llm (ChatOpenAI): Language model( with temperature: 0.0 ) used to evaluate the optimized prompt

    Returns:
        GraphState: Updated state with evaluation feedback
    """
    before_optimization_prompt = state["prompt"][-2]
    improved_prompt = state["prompt"][-1]

    prompt = PromptTemplate.from_template(
        """
    Now, let's evaluate the prompt. Follow these instructions:
    - 1. Comparision : read two prompts carefully and think about the difference between them.
    previous prompt: {before_optimization_prompt}
    revised prompt: {improved_prompt}

    - 2. Scoring: Give a score between 0 and 1 for 4 criteria. Show your the total score at the end.
    If yes add 1, else add 0.
      - 2.1) Is the revised prompt aligned with the user's intent? (yes or no)
      - 2.2) Is the revised prompt able to generate a better output than the previous prompt? (yes or no)
      - 2.3) Is the revised prompt well-structured? (yes or no)
      - 2.4) Is the revised prompt of flexible length? (yes or no)
    Sum up your score for each criterion: [ ]

    - 3. Writedown the justification for your score within 2 sentences.

    """
    )

    chain = prompt | llm | StrOutputParser()

    response = chain.invoke(
        {
            "before_optimization_prompt": before_optimization_prompt,
            "improved_prompt": improved_prompt,
        }
    )
    return {"ai_feedback": [response]}


# An additional path: CONTINUE
def should_continue(state: GraphState) -> str:
    """
    Determines whether to continue the optimization loop based on the user's feedback.

    Args:
        state (GraphState): Current state containing the user's feedback

    Returns:
        str: "FINISH" if the user wants to quit, "CONTINUE" otherwise
    """
    latest_human_feedback = state["human_feedback"][-1].strip()
    if latest_human_feedback == "q" or latest_human_feedback == "quit":
        return "FINISH"
    else:
        return "CONTINUE"


## Set up the Graph
# Memory Management
memory = MemorySaver()
workflow = StateGraph(GraphState)

# Node Configuration
workflow.add_node("analyze_prompt", analyze_prompt)
workflow.add_node("human", human_feedback)
workflow.add_node("test_prompt", test_optimized_prompt)
workflow.add_node("optimize_prompt", optimize_prompt)
workflow.add_node("evaluate_prompt", evaluate_prompt)

# Edge Configuration
# Add edges for the new flow
workflow.add_edge("analyze_prompt", "human")
workflow.add_edge("human", "test_prompt")
workflow.add_edge("test_prompt", "optimize_prompt")
workflow.add_edge("optimize_prompt", "evaluate_prompt")
workflow.add_conditional_edges(
    "evaluate_prompt", should_continue, {"CONTINUE": "human", "FINISH": END}
)

# Graph Compilation
workflow.set_entry_point("analyze_prompt")
graph = workflow.compile(checkpointer=memory)


# Visualize the graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig

# Configure execution parameters
config = RunnableConfig(
    recursion_limit=100, configurable={"thread_id": str(uuid.uuid4())}
)


initial_prompt = """우유가 넘어지면 뭘까? 넌센스 퀴즈야"""  # ←-Write your prompt here.


# Initialize graph state with input prompt
inputs = GraphState(prompt=[initial_prompt])

# Execute graph and stream updates
for event in graph.stream(inputs, config, stream_mode="updates"):
    for key, value in event.items():
        print(f"\n[ {key} ]\n")
        if key == "analyze_prompt":
            filtered_values = {k: v for k, v in value.items() if k != "prompt"}
        elif key == "human":
            filtered_values = {
                k: ("" if v[-1] in ["q", "quit"] else v[-1]) for k, v in value.items()
            }
        else:
            filtered_values = value
        for _, v in filtered_values.items():
            if isinstance(v, list):
                print(f"{v[-1]}")
            else:
                print(v)
    print("===" * 10, " STEPS ", "===" * 10)